# NHAMCS Scenario 2 — Opioids in the emergency department, 2018 to 2022

_Following a practice over time with repeated cross-sections. Level: second analysis. About 90 minutes._

**Before you start.** NHAMCS is a national probability sample of US emergency department visits:
86,864 visits abstracted from the records of about 200 hospitals a year, 2018 to 2022. Every record
carries a weight, the number of US visits it stands for, and **the MIP does not use weights**: every
figure in this notebook describes *the sampled visits*, not US emergency care. CDC codes are kept:
questions are 1 = Yes, 2 = No; checkbox items 0 = not marked, 1 = marked; Sex is 1 = Female,
2 = Male; unknown is 98. One row is one visit, and the same person may appear twice.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model). No
visit record ever reaches this notebook. Run the cells in order; each step says what to look for
and what result to expect.

**Learning goals.** Distinguish a drug given in the ED from a drug prescribed at discharge;
describe a trend across survey years and test it; relate a practice to patient factors with
cross-tabulations; fit a logistic regression that adjusts a trend for case mix; and discuss what a
repeated cross-sectional survey can say about change in practice.

**Clinical background.** Opioid analgesics are effective for acute severe pain and are commonly
given in EDs, by injection or by mouth. In the United States, ED prescriptions at discharge came
under scrutiny during the opioid epidemic, and from about 2016 guidelines and state laws pushed
towards shorter courses and non-opioid alternatives (NSAIDs, acetaminophen). NHAMCS records each
medication with a flag telling whether it was **given in the ED**, **prescribed at discharge**, or
both, and classifies drugs with the Multum therapeutic categories; the curators derived from these
the opioid, NSAID and acetaminophen flags used here. Drug classes come from a classification, not
from a review of each prescription: opioid-containing cough medicines and buprenorphine for
dependence count as opioids. Dose and duration are not recorded.

## 1. Connect to the MIP and open the NHAMCS pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("NHAMCS_ED")   # add version="v1" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
ed = dm.datasets.list()[0]              # one dataset: the five survey years together
display(dm)
display(ed)

In [ ]:
# The variable tree follows the visit: Survey administration, Arrival and visit timing, Patient,
# Expected source of payment, Triage and initial vital signs, Reason for visit, Injury, Diagnoses,
# Chronic conditions, Diagnostic services, Procedures, Medications, Vital signs after triage,
# Providers seen, Disposition, ED and hospital characteristics.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the datasets of the pathology
PCT, MIN = "{:.1f}", "{:.0f}"           # number formats of the tables: percentages, minutes


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the NHAMCS dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ed], variables=variables), filters=filters,
                        handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == POOLED]


def numeric_summary(result, variables):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def yes_percent(result, variable, level="Yes"):
    """Percentage of one level (default Yes) of a categorical variable in a describe result."""
    k = category_counts(result, variable)
    return float(k.loc[k["level"] == level, "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def rates_by(group, outcomes, levels=None, base_filter=None, event="1"):
    """% of visits with each outcome (code `event`) in every level of `group`: one describe per
    level for the counts of all outcomes, one Chi-squared Test per outcome (in .attrs)."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f).describe()
        row = {group.label: lab.get(c, c)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.attrs["chi2"], table.attrs["p"] = {}, {}
    for y in outcomes:
        f = F(group).is_not_null() & F(y).is_not_null()
        if levels:
            f = f & F(group).isin(list(levels))
        if base_filter is not None:
            f = f & base_filter
        chi = analysis([group, y], f, drop_missing=True).chi_square_test(x=group, y=y)
        table.attrs["chi2"][y.label] = chi.raw.get("chi2")
        table.attrs["p"][y.label] = chi.raw.get("p_value")
    return table


def band_rates(outcomes, bands, base_filter=None, event="1"):
    """% of visits with each outcome in bands of a variable: bands maps a label to a filter."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f).describe()
        row = {"band": label}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.1f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(6, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def anova_summary(result):
    """One-way ANOVA result -> F, p, n, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), table.get("n_obs"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Code | Label in the MIP | Tree location (under NHAMCS_ED) | Role | Values |
|---|---|---|---|---|
| `med_opioid_given` | Opioid given in ED | Medications / Drug classes | outcome; curation-derived | 0 No; 1 Yes |
| `med_opioid_rx` | Opioid prescribed at discharge | Medications / Drug classes | outcome; curation-derived | 0 No; 1 Yes |
| `med_opioid` | Opioid analgesic | Medications / Drug classes | any opioid (Step 1); curation-derived | 0 No; 1 Yes |
| `med_nsaid` | NSAID | Medications / Drug classes | comparison; curation-derived | 0 No; 1 Yes |
| `med_acetaminophen` | Acetaminophen or other non-opioid analgesic | Medications / Drug classes | comparison; curation-derived | 0 No; 1 Yes |
| `YEAR` | Survey year | Survey administration | trend; covariate | 2018 2018; 2019 2019; 2020 2020; 2021 2021; 2022 2022 |
| `AGER` | Age group | Patient | exposure | 1 Under 15 years; 2 15-24 years; 3 25-44 years; 4 45-64 years; 5 65-74 years; 6 75 years and over |
| `AGE` | Age | Patient | covariate | integer, 0-94 years |
| `SEX` | Sex | Patient | covariate | 1 Female; 2 Male |
| `PAINSCALE` | Pain scale | Triage and initial vital signs | exposure; cohort filter 0 to 10 | 12 codes, listed in the tree |
| `INJURY` | Injury-related visit | Injury | exposure; cohort filter 0, 1 | 0 No; 1 Yes; 2 Questionable injury status; 98 Unknown |
| `diag1_chapter` | Primary diagnosis, ICD-10 chapter | Diagnoses | exposure; curation-derived | 23 codes, listed in the tree |
| `disposition` | Disposition summary | Disposition | cohort filter (Step 6); curation-derived | 1 Treated and released (no other disposition marked); 2 Admitted to this hospital (directly or via observation); 3 Observation unit, then discharged; 4 Transferred (other hospital, psychiatric hospital, nursing home); 5 Left without being seen, before treatment complete, or against medical advice; 6 Died (dead on arrival or died in ED); 7 Other disposition only |
| `RACERETH` | Race/ethnicity (imputed) | Patient | exposure (extension) | 1 Non-Hispanic White; 2 Non-Hispanic Black; 3 Hispanic; 4 Non-Hispanic Other |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `WAITTIME`) is what the
csv column is called; you will see it in some result payloads.

In [ ]:
given      = dm.variables["Opioid given in ED"]
rx         = dm.variables["Opioid prescribed at discharge"]
any_opioid = dm.variables["Opioid analgesic"]
nsaid      = dm.variables["NSAID"]
acet       = dm.variables["Acetaminophen or other non-opioid analgesic"]
year       = dm.variables["Survey year"]
ager       = dm.variables["Age group"]
age        = dm.variables["Age"]
sex        = dm.variables["Sex"]
pain       = dm.variables["Pain scale"]
injury     = dm.variables["Injury-related visit"]
dispo      = dm.variables["Disposition summary"]
chapter    = dm.variables["Primary diagnosis, ICD-10 chapter"]
display(mip.to_frame([given, rx, any_opioid, nsaid, acet, year, ager, age, sex, pain, injury, dispo, chapter]))

## Step 1. How often, and where?

One `describe()` of the five medication flags on all visits; each is a checkbox-style variable,
0 = no, 1 = yes, recorded for every visit.

*Expected (86,864 visits):* an opioid was given in the ED in 11.7% of visits and prescribed at
discharge in 6.2% (any opioid 14.8%; some visits have both). NSAIDs were involved in 21.6% and
acetaminophen in 15.1%. Opioids are mostly an in-ED treatment; prescriptions to take home are half
as frequent.

In [ ]:
drugs = [given, rx, any_opioid, nsaid, acet]
res = analysis(drugs).describe()
use = pd.DataFrame({"% of visits": {v.label: yes_percent(res, v) for v in drugs}})
display(use.style.format(PCT))
ax = use["% of visits"].plot(kind="barh", color="#4C72B0", figsize=(7, 2.8))
ax.invert_yaxis(); ax.set_xlabel("% of visits"); ax.set_title("Analgesics in the sampled ED visits, 2018 to 2022")
plt.tight_layout()
plt.show()

## Step 2. The trend

The `rates_by` helper cross-tabulates several outcomes against one grouping variable at once: one
`describe()` per survey year for the counts of the four flags, then one *Chi-squared Test* per
flag.

*Expected:*

| Year | Opioid prescribed at discharge | Opioid given in ED | NSAID | Acetaminophen |
|---|---|---|---|---|
| 2018 | 7.4% | 12.9% | 23.4% | 13.7% |
| 2019 | 6.3% | 11.8% | 22.8% | 14.1% |
| 2020 | 6.0% | 12.0% | 20.1% | 15.4% |
| 2021 | 5.4% | 11.1% | 20.2% | 14.9% |
| 2022 | 5.4% | 10.7% | 20.7% | 17.7% |

Prescribing fell by a quarter in relative terms (chi-square 82, p < 0.001); in-ED administration
fell less (chi-square 49); acetaminophen rose (chi-square 134). This is the national
de-prescribing movement seen in a sample.

In [ ]:
by_year = rates_by(year, [rx, given, nsaid, acet]).set_index("Survey year")
display(by_year.style.format({c: PCT for c in by_year.columns if c != "n"}))
for label, chi2 in by_year.attrs["chi2"].items():
    print(f"{label}: chi2 = {chi2:.0f}, p = {fmt_p(by_year.attrs['p'][label])}")

ax = by_year.drop(columns="n").plot(marker="o", figsize=(7, 3.5))
ax.set_ylim(0); ax.set_ylabel("% of visits"); ax.set_xlabel(""); ax.set_title("Analgesics by survey year")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## Step 3. To whom: age

The two opioid flags by age group.

*Expected:* given in 2.3% of visits under 15 years, 8.0% at 15 to 24, 13.2% at 25 to 44, 17.3% at
45 to 64, 15.5% at 65 to 74, 13.4% at 75 and over (p < 0.001). Prescribed at discharge: 0.7%, 4.3%,
8.0%, 8.9%, 7.4%, 5.9%. Children almost never receive opioids; middle-aged adults most often.

In [ ]:
by_age = rates_by(ager, [given, rx]).set_index("Age group")
display(by_age.style.format({c: PCT for c in by_age.columns if c != "n"}))
for label, chi2 in by_age.attrs["chi2"].items():
    print(f"{label}: chi2 = {chi2:.0f}, p = {fmt_p(by_age.attrs['p'][label])}")
ax = by_age.drop(columns="n").plot(kind="bar", figsize=(7, 3.2), color=["#4C72B0", "#DD8452"], rot=0)
ax.set_ylabel("% of visits"); ax.set_xlabel(""); ax.set_title("Opioids by age group")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Step 4. To whom: pain and injury

The pain scale is recorded for 52,106 visits (0 to 10; 98 = unknown, empty for the rest). The
`band_rates` helper takes one filter per band, one `describe()` each. Then the cross-tabulation
of the discharge prescription by injury status, on all visits.

*Expected:* opioid given in 3.4% of visits with pain 0, 6.6% with pain 1 to 3, 11.8% with 4 to 6,
26.3% with 7 to 10; prescribed at discharge 1.4%, 4.1%, 6.6%, 13.5%. Injury-related visits:
prescribed in 9.1% versus 5.0% of non-injury visits (p < 0.001). Prescribing follows pain
intensity and injury, as it should. Note that pain is unrecorded in 40% of visits, and those
visits are not random (children, altered patients).

In [ ]:
bands = {"0, no pain": F(pain) == "0",
         "1 to 3": F(pain).isin(["1", "2", "3"]),
         "4 to 6": F(pain).isin(["4", "5", "6"]),
         "7 to 10": F(pain).isin(["7", "8", "9", "10"])}
by_pain = band_rates([given, rx], bands).set_index("band")
display(by_pain.style.format({c: PCT for c in by_pain.columns if c != "n"}))
print(f"Visits with a recorded pain score: {int(by_pain['n'].sum()):,}")

by_injury = crosstab(injury, rx)
show_crosstab(by_injury, "Opioid prescribed at discharge, by injury status")

## Step 5. By diagnosis

The cross-tabulation of the in-ED opioid by the ICD-10 chapter of the primary diagnosis: 23
chapters, so 23 `describe()` calls and one chi-square. Chapters with fewer than 10 visits come
back empty (the platform's privacy threshold) and are left out.

*Expected highs:* blood and immune disorders 30% (a small chapter that includes sickle-cell
crises, a classic indication), neoplasms 28%, digestive 24% (biliary and abdominal pain; renal
colic sits in genitourinary at 21%), musculoskeletal 21%, injuries 15%. *Lows:* mental disorders,
respiratory, eye and ear, each under 4%; pregnancy 6%.

In [ ]:
by_chapter = crosstab(chapter, given)
print(f"Chi-squared Test: chi2 = {by_chapter.attrs['chi2']:.0f}, p = {fmt_p(by_chapter.attrs['p'])}; "
      f"{len(by_chapter)} chapters with at least 10 visits")
ranked = by_chapter.sort_values("% Yes", ascending=False).set_index(chapter.label)
display(ranked[["n", "% Yes"]].style.format({"% Yes": PCT}))
ax = ranked["% Yes"].iloc[::-1].plot(kind="barh", color="#4C72B0", figsize=(8, 6))
ax.set_xlabel("% of visits with an opioid given in the ED"); ax.set_ylabel(""); ax.set_title("Opioid given in the ED, by diagnosis chapter")
plt.tight_layout()
plt.show()

## Step 6. Adjusting the trend for case mix

Visits treated and released with a known injury status, and a *Logistic Regression* of the
discharge prescription on age, sex, injury and survey year. The platform compares each category
with its first code: female, no injury, 2018.

*Expected (63,099 visits, 7.4% prescribed):* odds ratios per year of age 1.021 (1.019 to 1.022);
male 0.97 (not significant); injury 1.81 (1.71 to 1.93); 2019 0.85, 2020 0.79, 2021 0.69, 2022
0.69, all p < 0.001. Adjusting for age, sex and injury does not explain the decline: in 2021 and
2022 the odds of leaving the ED with an opioid prescription were 31% lower than in 2018 for the
same kind of patient.

In [ ]:
released = (F(dispo) == "1") & F(injury).isin(["0", "1"])     # treated and released, injury status known
covariates = [age, sex, injury, year]
model = analysis([rx] + covariates, released, drop_missing=True).logistic_regression(
    x=covariates, y=rx, positive_class="1")
print(f"{model.raw['summary']['n_obs']:,} visits in the model")
ors = odds_ratio_table(model, covariates)
show_odds_ratios(ors, "Odds of an opioid prescription at discharge, treated-and-released visits")

## Step 7. State the result

A model answer:

> In 86,864 sampled US ED visits from 2018 to 2022, an opioid was given in the ED in 12% of visits
> and prescribed at discharge in 6%. Prescribing at discharge fell from 7.4% in 2018 to 5.4% in
> 2022, a decline that persists after adjustment for age, sex and injury (odds ratio 0.69 for 2022
> versus 2018 among patients treated and released), while acetaminophen use rose. Opioids were
> given mainly to adults in severe pain (26% of visits with pain 7 to 10 versus 3% with pain 0) and
> after injury, and rarely to children (2%). The sample is unweighted and the drug classification
> is broad, so these are indicative figures of a practice change, consistent with US
> de-prescribing policies.

**Questions for discussion.** Repeated cross-sections show that prescribing fell: can they show
whether patients' pain was less well treated? The opioid flag counts cough medicines and
buprenorphine: in which direction does that bias the percentages, and does it affect the trend?
Pain is recorded in 60% of visits: how could the missing 40% distort Step 4?

**Optional extension.** Race and ethnicity: cross-tabulate the imputed race/ethnicity variable
against the discharge prescription (expected 7.2% for non-Hispanic White patients versus about
5% for the other three groups), then check the age and pain distributions of each group before
interpreting. Disparities in pain treatment are a known research topic in US emergency medicine.

In [ ]:
race = dm.variables["Race/ethnicity (imputed)"]
show_crosstab(crosstab(race, rx), "Opioid prescribed at discharge, by race/ethnicity")